# MNE official Sleep-EDF sleep staging reproduction

This notebook reproduces MNE-Python's official "Sleep stage classification from polysomnography (PSG) data" tutorial for the requested minimal case: Sleep Cassette subjects 0 and 1, recording 1. Subject 0 / night 1 is the training set and subject 1 / night 1 is the test set.

Method lock: no time-domain band-pass or notch filter, ICA, wavelets, SMOTE, class_weight, extra channels, or alternative models. The 0.5--30 Hz interval is the PSD analysis range only.


In [ ]:
from pathlib import Path
import json
import os
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from mne.datasets.sleep_physionet.age import fetch_data
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results" / "reproduction"
FIGURES_DIR = PROJECT_ROOT / "figures" / "reproduction"
MNE_CACHE_DIR = PROJECT_ROOT / "data" / "mne_data"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MNE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["MNE_DATA"] = str(MNE_CACHE_DIR)

print("Python:", sys.version.split()[0])
print("MNE:", mne.__version__)
print("MNE cache:", MNE_CACHE_DIR.relative_to(PROJECT_ROOT).as_posix())
print("scikit-learn:", __import__("sklearn").__version__)


## 1. Fetch Sleep Cassette data on demand

The official MNE fetcher returns one PSG EDF and one expert Hypnogram EDF per requested subject/recording. No whole-database download is used.


In [ ]:
SUBJECTS = [0, 1]
RECORDING = [1]
files_by_subject = fetch_data(subjects=SUBJECTS, recording=RECORDING, path=MNE_CACHE_DIR, verbose=True)

file_records = []
for subject, files in zip(SUBJECTS, files_by_subject):
    if len(files) != 2:
        raise RuntimeError(f"Expected PSG and Hypnogram for subject {subject}, got {files}")
    psg_path, hypnogram_path = map(Path, files)
    if not psg_path.name.endswith("-PSG.edf"):
        raise AssertionError(f"Unexpected PSG filename: {psg_path}")
    if not hypnogram_path.name.endswith("-Hypnogram.edf"):
        raise AssertionError(f"Unexpected Hypnogram filename: {hypnogram_path}")
    file_records.append({
        "subject": subject,
        "recording": RECORDING[0],
        "psg_edf": psg_path.resolve().relative_to(PROJECT_ROOT).as_posix(),
        "hypnogram_edf": hypnogram_path.resolve().relative_to(PROJECT_ROOT).as_posix(),
        "psg_exists": psg_path.is_file(),
        "hypnogram_exists": hypnogram_path.is_file(),
    })

if not all(r["psg_exists"] and r["hypnogram_exists"] for r in file_records):
    raise FileNotFoundError(file_records)

mne_cache_path = Path(file_records[0]["psg_edf"]).parent.parent.as_posix()
fetch_summary = {
    "mne_config_MNE_DATA": MNE_CACHE_DIR.relative_to(PROJECT_ROOT).as_posix(),
    "actual_cache_path_from_fetched_files": mne_cache_path,
    "subjects": SUBJECTS,
    "recording": RECORDING,
    "files": file_records,
}
(RESULTS_DIR / "fetched_files.json").write_text(json.dumps(fetch_summary, indent=2), encoding="utf-8")
print(json.dumps(fetch_summary, indent=2))


## 2. Read EDF files and create official 30-second epochs

Only W/1/2/3/4/R are mapped. Stage 3 and Stage 4 share event code 4. The annotation crop follows the official tutorial: retain the approximately 30 minutes of Wake immediately before the first scored sleep annotation and after the last scored sleep annotation.


In [6]:
annotation_desc_2_event_id = {
    "Sleep stage W": 1,
    "Sleep stage 1": 2,
    "Sleep stage 2": 3,
    "Sleep stage 3": 4,
    "Sleep stage 4": 4,
    "Sleep stage R": 5,
}
event_id = {
    "Sleep stage W": 1,
    "Sleep stage 1": 2,
    "Sleep stage 2": 3,
    "Sleep stage 3/4": 4,
    "Sleep stage R": 5,
}

def load_epochs(psg_path, hypnogram_path):
    raw = mne.io.read_raw_edf(
        psg_path,
        stim_channel="Event marker",
        infer_types=True,
        preload=True,
        verbose="error",
    )
    annotations = mne.read_annotations(hypnogram_path)
    annotations.crop(annotations[1]["onset"] - 30 * 60, annotations[-2]["onset"] + 30 * 60)
    raw.set_annotations(annotations, emit_warning=False)
    events, _ = mne.events_from_annotations(
        raw, event_id=annotation_desc_2_event_id, chunk_duration=30.0
    )
    tmax = 30.0 - 1.0 / raw.info["sfreq"]
    epochs = mne.Epochs(
        raw=raw,
        events=events,
        event_id=event_id,
        tmin=0.0,
        tmax=tmax,
        baseline=None,
        preload=True,
        verbose=False,
    )
    return raw, annotations, events, epochs

raw_train, annot_train, events_train, epochs_train = load_epochs(
    PROJECT_ROOT / file_records[0]["psg_edf"], PROJECT_ROOT / file_records[0]["hypnogram_edf"]
)
raw_test, annot_test, events_test, epochs_test = load_epochs(
    PROJECT_ROOT / file_records[1]["psg_edf"], PROJECT_ROOT / file_records[1]["hypnogram_edf"]
)

eeg_channel_names = [
    ch for ch, kind in zip(epochs_train.ch_names, epochs_train.get_channel_types())
    if kind == "eeg"
]
if eeg_channel_names != ["Fpz-Cz", "Pz-Oz"]:
    raise AssertionError(f"Expected the two Sleep-EDF EEG channels, got {eeg_channel_names}")

print("Training channel names:", epochs_train.ch_names)
print("Training EEG channels used by picks='eeg':", eeg_channel_names)
print("Training sfreq:", epochs_train.info["sfreq"])
print("Test sfreq:", epochs_test.info["sfreq"])
print("Epoch duration (s):", epochs_train.times[-1] + 1.0 / epochs_train.info["sfreq"])
print("Training epochs:", len(epochs_train))
print("Test epochs:", len(epochs_test))
print("Training event counts:", {name: int((epochs_train.events[:, 2] == code).sum()) for name, code in event_id.items()})
print("Test event counts:", {name: int((epochs_test.events[:, 2] == code).sum()) for name, code in event_id.items()})


Used Annotations descriptions: ['Sleep stage 1', 'Sleep stage 2', 'Sleep stage 3', 'Sleep stage 4', 'Sleep stage R', 'Sleep stage W']
Used Annotations descriptions: ['Sleep stage 1', 'Sleep stage 2', 'Sleep stage 3', 'Sleep stage 4', 'Sleep stage R', 'Sleep stage W']
Training channel names: ['Fpz-Cz', 'Pz-Oz', 'horizontal', 'oro-nasal', 'submental', 'rectal', 'Event marker']
Training EEG channels used by picks='eeg': ['Fpz-Cz', 'Pz-Oz']
Training sfreq: 100.0
Test sfreq: 100.0
Epoch duration (s): 30.0
Training epochs: 841
Test epochs: 1103
Training event counts: {'Sleep stage W': 188, 'Sleep stage 1': 58, 'Sleep stage 2': 250, 'Sleep stage 3/4': 220, 'Sleep stage R': 125}
Test event counts: {'Sleep stage W': 157, 'Sleep stage 1': 109, 'Sleep stage 2': 562, 'Sleep stage 3/4': 105, 'Sleep stage R': 170}


## 3. Official relative PSD band-power features

The PSD range is 0.5--30 Hz. Features are normalized relative band powers for delta (0.5--4.5), theta (4.5--8.5), alpha (8.5--11.5), sigma (11.5--15.5), and beta (15.5--30) Hz. Because picks="eeg", the feature dimension is 2 EEG channels × 5 bands = 10.


In [8]:
FREQ_BANDS = {
    "delta": [0.5, 4.5],
    "theta": [4.5, 8.5],
    "alpha": [8.5, 11.5],
    "sigma": [11.5, 15.5],
    "beta": [15.5, 30.0],
}

def eeg_power_band(epochs):
    spectrum = epochs.compute_psd(picks="eeg", fmin=0.5, fmax=30.0)
    psds, freqs = spectrum.get_data(return_freqs=True)
    psds /= np.sum(psds, axis=-1, keepdims=True)
    X = []
    for fmin, fmax in FREQ_BANDS.values():
        psds_band = psds[:, :, (freqs >= fmin) & (freqs < fmax)].mean(axis=-1)
        X.append(psds_band.reshape(len(psds), -1))
    return np.concatenate(X, axis=1)

X_train_sanity = eeg_power_band(epochs_train)
X_test_sanity = eeg_power_band(epochs_test)
expected_feature_dim = len(eeg_channel_names) * len(FREQ_BANDS)
if X_train_sanity.shape[1] != expected_feature_dim or X_test_sanity.shape[1] != expected_feature_dim:
    raise AssertionError((X_train_sanity.shape, X_test_sanity.shape, expected_feature_dim))
print("Feature shapes:", X_train_sanity.shape, X_test_sanity.shape)
print("Expected feature dimension:", expected_feature_dim)


    Using multitaper spectrum estimation with 7 DPSS windows
    Using multitaper spectrum estimation with 7 DPSS windows
Feature shapes: (841, 10) (1103, 10)
Expected feature dimension: 10


## 4. Official Random Forest baseline and saved outputs


In [10]:
pipe = make_pipeline(
    FunctionTransformer(eeg_power_band, validate=False),
    RandomForestClassifier(n_estimators=100, random_state=241),
)

y_train = epochs_train.events[:, 2]
y_test = epochs_test.events[:, 2]
pipe.fit(epochs_train, y_train)
y_pred = pipe.predict(epochs_test)

labels = list(event_id.values())
target_names = list(event_id.keys())
acc = float(accuracy_score(y_test, y_pred))
macro_f1 = float(f1_score(y_test, y_pred, labels=labels, average="macro", zero_division=0))
report_dict = classification_report(
    y_test, y_pred, labels=labels, target_names=target_names, output_dict=True, zero_division=0
)
report_text = classification_report(
    y_test, y_pred, labels=labels, target_names=target_names, zero_division=0
)
cm = confusion_matrix(y_test, y_pred, labels=labels)

pd.DataFrame(report_dict).T.to_csv(RESULTS_DIR / "classification_report.csv")
(RESULTS_DIR / "classification_report.txt").write_text(report_text, encoding="utf-8")
(RESULTS_DIR / "classification_report.json").write_text(json.dumps(report_dict, indent=2), encoding="utf-8")
pd.DataFrame(cm, index=target_names, columns=target_names).to_csv(RESULTS_DIR / "confusion_matrix.csv")
np.save(RESULTS_DIR / "confusion_matrix.npy", cm)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm, cmap="Blues")
ax.set(
    xticks=range(len(target_names)),
    yticks=range(len(target_names)),
    xticklabels=target_names,
    yticklabels=target_names,
    xlabel="Predicted",
    ylabel="True",
    title="MNE official reproduction confusion matrix",
)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center")
fig.colorbar(im, ax=ax, label="Epoch count")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "confusion_matrix.png", dpi=180)
plt.close(fig)

sanity_checks = {
    "mne_version": mne.__version__,
    "sklearn_version": __import__("sklearn").__version__,
    "train_subject": 0,
    "test_subject": 1,
    "recording": 1,
    "channel_names": epochs_train.ch_names,
    "eeg_channels_picks_eeg": eeg_channel_names,
    "sfreq_train_hz": float(epochs_train.info["sfreq"]),
    "sfreq_test_hz": float(epochs_test.info["sfreq"]),
    "epoch_duration_s": float(epochs_train.times[-1] + 1.0 / epochs_train.info["sfreq"]),
    "classes": target_names,
    "n_classes": len(target_names),
    "n_train_epochs": int(len(epochs_train)),
    "n_test_epochs": int(len(epochs_test)),
    "train_class_counts": {name: int((y_train == code).sum()) for name, code in event_id.items()},
    "test_class_counts": {name: int((y_test == code).sum()) for name, code in event_id.items()},
    "feature_shape_train": list(X_train_sanity.shape),
    "feature_shape_test": list(X_test_sanity.shape),
    "expected_feature_dim": int(expected_feature_dim),
    "psd_range_hz": [0.5, 30.0],
    "frequency_bands_hz": FREQ_BANDS,
    "accuracy": acc,
    "macro_f1": macro_f1,
    "confusion_matrix": cm.tolist(),
    "output_paths": {
        "classification_report": (RESULTS_DIR / "classification_report.csv").relative_to(PROJECT_ROOT).as_posix(),
        "confusion_matrix": (RESULTS_DIR / "confusion_matrix.csv").relative_to(PROJECT_ROOT).as_posix(),
        "confusion_matrix_figure": (FIGURES_DIR / "confusion_matrix.png").relative_to(PROJECT_ROOT).as_posix(),
    },
}
(RESULTS_DIR / "metrics.json").write_text(json.dumps(sanity_checks, indent=2), encoding="utf-8")

print(f"Accuracy: {acc:.10f}")
print(f"Macro-F1: {macro_f1:.10f}")
print(report_text)
print("Confusion matrix:\n", cm)


    Using multitaper spectrum estimation with 7 DPSS windows
    Using multitaper spectrum estimation with 7 DPSS windows
Accuracy: 0.6872166818
Macro-F1: 0.5743399516
                 precision    recall  f1-score   support

  Sleep stage W       0.48      0.97      0.64       157
  Sleep stage 1       0.22      0.27      0.24       109
  Sleep stage 2       0.94      0.80      0.86       562
Sleep stage 3/4       0.78      0.96      0.86       105
  Sleep stage R       0.58      0.17      0.26       170

       accuracy                           0.69      1103
      macro avg       0.60      0.63      0.57      1103
   weighted avg       0.73      0.69      0.68      1103

Confusion matrix:
 [[152   0   1   4   0]
 [ 68  29   8   1   3]
 [ 44  30 447  23  18]
 [  0   0   4 101   0]
 [ 52  72  17   0  29]]


## Interpretation

The official gallery reference is approximately Accuracy 0.6872 and Macro-F1 0.57, with N1 and REM substantially harder than Wake, N2, and merged N3/4. Small numerical differences are acceptable; the sanity checks above verify the requested processing logic and scale.
